# NB29 · Performance & Memory

`advanced/` · 20 questions · the last one is a story project

Measure first, then optimise. You will time code with `timeit` and
`perf_counter`, profile it with `cProfile`, choose containers by their
real cost, save memory with generators, `__slots__` and `array`, and see
how Python frees objects (reference counting, the cycle collector and
`weakref`). Timings differ on every machine, so the tasks print counts,
results and yes/no comparisons with a huge margin instead of raw times.

## You'll need

- NB19: Big-O
- NB28: testing

## You'll practise

- Measuring with timeit, perf_counter and cProfile
- Costs of built-in operations; choosing the right container
- join vs += for strings; generators to save memory
- __slots__, caching, local variable lookups
- References, reference counting, gc, cycles, weakref
- sys.getsizeof and its limits; array vs list for numbers

## How it works

Write your code in the cell under each question, run it, and compare with **Expected output**. Open **Solution** only after a real attempt.

Questions get longer as you go: one-liners first, a small story project last. Every solution only uses what this notebook and the ones before it teach.

---

## Warm-up · one line each

### Q01 · Build

`timeit.timeit(func, number=N)` runs `func` exactly `N` times. Prove it:
time a lambda that appends `1` to the list `calls` 500 times, then print
how many items `calls` holds.

**Expected output:**

```text
500
```

<details><summary><b>Solution</b></summary>

```python
import timeit
timeit.timeit(lambda: calls.append(1), number=500)
print(len(calls))
```

</details>

In [ ]:
calls = []

# your code here


### Q02 · Real-life

A mailing list has 100,000 subscriber numbers, stored both as a list and
as a set. Use `timeit` (200 runs each, passing `globals=globals()`) to
time the check `99_999 in ...` on both. Print `True` if the list version
took more than 50 times longer than the set version.

**Expected output:**

```text
Set at least 50x faster: True
```

<details><summary><b>Solution</b></summary>

```python
import timeit
list_time = timeit.timeit("99_999 in numbers", globals=globals(), number=200)
set_time = timeit.timeit("99_999 in number_set", globals=globals(), number=200)
print("Set at least 50x faster:", list_time > set_time * 50)
```

</details>

In [ ]:
numbers = list(range(100_000))
number_set = set(numbers)

# your code here


### Q03 · Puzzle

`sys.getsizeof` only measures the container itself, not what it points to.
Print `True` if `light` and `heavy` report the same size, then print the
total number of numbers stored inside `heavy`.

**Expected output:**

```text
True
2000
```

<details><summary><b>Solution</b></summary>

```python
import sys
print(sys.getsizeof(light) == sys.getsizeof(heavy))
print(len(heavy[0]) + len(heavy[1]))
```

</details>

In [ ]:
light = [None, None]
heavy = [list(range(1000)), list(range(1000))]

# your code here


### Q04 · Real-life

Cache a slow recursive `fib(n)` with `functools.cache`. Print `fib(30)`,
then print the cache statistics line (`cache_info()`).

**Expected output:**

```text
832040
CacheInfo(hits=28, misses=31, maxsize=None, currsize=31)
```

<details><summary><b>Solution</b></summary>

```python
from functools import cache

@cache
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print(fib(30))
print(fib.cache_info())
```

</details>

In [ ]:
# your code here


### Q05 · Build

A `Session` object is created and a weak reference `ref` to it is made
with `weakref.ref`. Print whether `ref()` is the session, delete the
only normal reference, then print what `ref()` returns now.

**Expected output:**

```text
True
None
```

<details><summary><b>Solution</b></summary>

```python
import weakref
ref = weakref.ref(session)
print(ref() is session)
del session
print(ref())
```

</details>

In [ ]:
class Session:
    pass

session = Session()

# your code here


---

## Combine · 2 to 4 lines

### Q06 · Real-life

A report needs the squares of the first 100,000 numbers, once as a list
comprehension and once as a generator expression. Print `True` if the
generator object is more than 100 times smaller than the list (use
`sys.getsizeof`), and `True` if both give the same `sum`.

**Expected output:**

```text
True
True
```

<details><summary><b>Solution</b></summary>

```python
import sys
squares_list = [n * n for n in range(100_000)]
squares_gen = (n * n for n in range(100_000))
print(sys.getsizeof(squares_gen) * 100 < sys.getsizeof(squares_list))
print(sum(squares_list) == sum(squares_gen))
```

</details>

In [ ]:
# your code here


### Q07 · Output-match

A `Resource` prints `closed` the moment it is destroyed. Two names point
at the same one. Delete `first` and print `still alive`, then delete
`second` and print `done`. Python frees the object as soon as its last
reference disappears.

**Expected output:**

```text
still alive
closed
done
```

<details><summary><b>Solution</b></summary>

```python
del first
print("still alive")
del second
print("done")
```

</details>

In [ ]:
class Resource:
    def __del__(self):
        print("closed")

first = Resource()
second = first

# your code here


### Q08 · Build

Write a class `Point` that uses `__slots__ = ("x", "y")` and takes `x`
and `y` in `__init__`. Print whether a `Point` has a `__dict__`, then
try to set `p.z = 3` and print `no new attributes` when that fails.

**Expected output:**

```text
False
no new attributes
```

<details><summary><b>Solution</b></summary>

```python
class Point:
    __slots__ = ("x", "y")

    def __init__(self, x, y):
        self.x = x
        self.y = y

p = Point(1, 2)
print(hasattr(p, "__dict__"))
try:
    p.z = 3
except AttributeError:
    print("no new attributes")
```

</details>

In [ ]:
# your code here


### Q09 · Puzzle

Store the sensor readings (all between 0 and 65535) in an `array` with
typecode `"H"`. Print the bytes per item, the number of items, and the
total bytes (items times item size). Then try to append `70000`, which
does not fit, and print `too big` when that fails.

**Expected output:**

```text
2 100000 200000
too big
```

<details><summary><b>Solution</b></summary>

```python
from array import array
packed = array("H", readings)
print(packed.itemsize, len(packed), packed.itemsize * len(packed))
try:
    packed.append(70000)
except OverflowError:
    print("too big")
```

</details>

In [ ]:
readings = [n % 1000 for n in range(100_000)]

# your code here


### Q10 · Output-match

Build one text line from the integers in `temps` separated by `, `
using `join` (not repeated `+=`), print it, then print its length.

**Expected output:**

```text
12, 15, 11, 19, 14
18
```

<details><summary><b>Solution</b></summary>

```python
line = ", ".join(str(t) for t in temps)
print(line)
print(len(line))
```

</details>

In [ ]:
temps = [12, 15, 11, 19, 14]

# your code here


---

## Small real tasks · 5 to 8 lines

### Q11 · Real-life

Write `has_duplicate_slow(items)` using two nested loops over positions
and `has_duplicate_fast(items)` using a set. Time one call of each on
`ids` with `timeit.timeit(lambda: ..., number=1)`. Print both results
and then `True` if the fast version is at least 10 times faster.

**Expected output:**

```text
True True
Fast at least 10x faster: True
```

<details><summary><b>Solution</b></summary>

```python
import timeit

def has_duplicate_slow(items):
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            if items[i] == items[j]:
                return True
    return False

def has_duplicate_fast(items):
    return len(set(items)) < len(items)

slow_time = timeit.timeit(lambda: has_duplicate_slow(ids), number=1)
fast_time = timeit.timeit(lambda: has_duplicate_fast(ids), number=1)
print(has_duplicate_slow(ids), has_duplicate_fast(ids))
print("Fast at least 10x faster:", slow_time > fast_time * 10)
```

</details>

In [ ]:
ids = list(range(3000)) + [1500]

# your code here


### Q12 · Real-life

Profile `fib(20)` with `cProfile.Profile().runcall(...)`. Wrap the
profiler in `pstats.Stats` and loop over `stats.stats.items()`: each key
is `(file, line, name)` and each value starts with `(primitive_calls,
total_calls, ...)`. Print how many times `fib` was called in total.

**Expected output:**

```text
fib calls: 21891
```

<details><summary><b>Solution</b></summary>

```python
import cProfile
import pstats

profiler = cProfile.Profile()
profiler.runcall(fib, 20)
stats = pstats.Stats(profiler)
for (filename, line, name), values in stats.stats.items():
    if name == "fib":
        print("fib calls:", values[1])
```

</details>

In [ ]:
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

# your code here


### Q13 · Real-life

A job queue is drained from the front. Write `drain_list(items)` using
`list.pop(0)` and `drain_deque(items)` using a `collections.deque` and
`popleft()`; both return the items in the order they were taken. Print
the first three items of the deque result, `True` if both orders are
equal, and `True` if the deque version is more than 5 times faster.

**Expected output:**

```text
[0, 1, 2]
True
True
```

<details><summary><b>Solution</b></summary>

```python
import timeit
from collections import deque

def drain_list(items):
    pending = list(items)
    order = []
    while pending:
        order.append(pending.pop(0))
    return order

def drain_deque(items):
    pending = deque(items)
    order = []
    while pending:
        order.append(pending.popleft())
    return order

list_time = timeit.timeit(lambda: drain_list(jobs), number=1)
deque_time = timeit.timeit(lambda: drain_deque(jobs), number=1)
print(drain_deque(jobs)[:3])
print(drain_list(jobs) == drain_deque(jobs))
print(list_time > deque_time * 5)
```

</details>

In [ ]:
jobs = list(range(40_000))

# your code here


### Q14 · Real-life

`tax_rate(country)` is a slow lookup, counted in `lookups["count"]`.
Cache it with `functools.lru_cache(maxsize=None)`, call it for every
country in `orders`, and print how many real lookups happened, then
the `hits` and `misses` from `cache_info()`. Finally clear the cache
and print its `currsize`.

**Expected output:**

```text
Real lookups: 3
Hits: 4 Misses: 3
Size after clear: 0
```

<details><summary><b>Solution</b></summary>

```python
from functools import lru_cache

tax_rate = lru_cache(maxsize=None)(tax_rate)
for country in orders:
    tax_rate(country)
print("Real lookups:", lookups["count"])
info = tax_rate.cache_info()
print("Hits:", info.hits, "Misses:", info.misses)
tax_rate.cache_clear()
print("Size after clear:", tax_rate.cache_info().currsize)
```

</details>

In [ ]:
RATES = {"DE": 0.19, "FR": 0.20, "ES": 0.21}
lookups = {"count": 0}
orders = ["DE", "FR", "DE", "DE", "FR", "ES", "DE"]

def tax_rate(country):
    lookups["count"] += 1
    return RATES[country]

# your code here


### Q15 · Real-life

Image loading is expensive, so `load(name)` should reuse an image that
is still in use but must not keep unused ones alive. Back it with a
`weakref.WeakValueDictionary` named `cache` and record every real
creation in `created`. Load `"logo"` twice, print whether both are the
same object and how many were created. Delete both names and print the
cache size. Load `"logo"` once more and print how many were created.

**Expected output:**

```text
True 1
0
2
```

<details><summary><b>Solution</b></summary>

```python
import weakref

cache = weakref.WeakValueDictionary()

def load(name):
    image = cache.get(name)
    if image is None:
        image = Image(name)
        created.append(name)
        cache[name] = image
    return image

a = load("logo")
b = load("logo")
print(a is b, len(created))
del a, b
print(len(cache))
load("logo")
print(len(created))
```

</details>

In [ ]:
class Image:
    def __init__(self, name):
        self.name = name

created = []

# your code here


---

## Puzzles & builds · 8 to 15 lines

### Q16 · Build

Write a `Timer` class that works in a `with` block: `__enter__` stores
`time.perf_counter()` and returns the timer, `__exit__` stores
`self.elapsed`. Use it to time `fib(27)` (plain) and `fib_cached(27)`
(with `functools.cache`). Print `True` if the results are equal and
`True` if the cached run was more than 100 times faster.

**Expected output:**

```text
True
True
```

<details><summary><b>Solution</b></summary>

```python
import time
from functools import cache

class Timer:
    def __enter__(self):
        self.start = time.perf_counter()
        return self

    def __exit__(self, *exc_info):
        self.elapsed = time.perf_counter() - self.start

@cache
def fib_cached(n):
    return n if n < 2 else fib_cached(n - 1) + fib_cached(n - 2)

with Timer() as slow:
    result_slow = fib(27)
with Timer() as fast:
    result_fast = fib_cached(27)
print(result_slow == result_fast)
print(slow.elapsed > fast.elapsed * 100)
```

</details>

In [ ]:
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

# your code here


### Q17 · Real-life

Count the `ERROR` lines of a log file with 50,000 lines in two ways:
`count_errors_list(path)` reads everything with `readlines()`,
`count_errors_stream(path)` loops over the open file line by line. Write
the log into a temporary directory (every 10th line is an `ERROR`).
Measure each function's peak memory with `tracemalloc` and print both
counts, then `True` if the streaming peak times 20 is still smaller than
the list peak.

**Expected output:**

```text
5000 5000
True
```

<details><summary><b>Solution</b></summary>

```python
import os
import tempfile
import tracemalloc

def count_errors_list(path):
    with open(path) as f:
        lines = f.readlines()
    return sum(1 for line in lines if " ERROR " in line)

def count_errors_stream(path):
    with open(path) as f:
        return sum(1 for line in f if " ERROR " in line)

def peak_of(func, path):
    tracemalloc.start()
    result = func(path)
    peak = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()
    return result, peak

with tempfile.TemporaryDirectory() as folder:
    path = os.path.join(folder, "server.log")
    with open(path, "w") as f:
        for i in range(50_000):
            level = "ERROR" if i % 10 == 0 else "INFO"
            f.write(f"2024-05-01 12:00:00 {level} request {i} finished\n")
    list_count, list_peak = peak_of(count_errors_list, path)
    stream_count, stream_peak = peak_of(count_errors_stream, path)
print(list_count, stream_count)
print(stream_peak * 20 < list_peak)
```

</details>

In [ ]:
# your code here


### Q18 · Puzzle

A lying benchmark: `slow_square` is cached, so timing it 1000 times
only measures the cache. Time `slow_square(12)` 1000 times with
`timeit` and print `Naive calls:` and the number of real calls. Reset
the counter, then time an honest version (a function that calls
`slow_square.cache_clear()` first, then `slow_square(12)`), also 1000
times, and print `Honest calls:` and the count.

**Expected output:**

```text
Naive calls: 1
Honest calls: 1000
```

<details><summary><b>Solution</b></summary>

```python
import timeit

timeit.timeit(lambda: slow_square(12), number=1000)
print("Naive calls:", calls["n"])

calls["n"] = 0

def honest():
    slow_square.cache_clear()
    return slow_square(12)

timeit.timeit(honest, number=1000)
print("Honest calls:", calls["n"])
```

</details>

In [ ]:
from functools import lru_cache

calls = {"n": 0}

@lru_cache(maxsize=None)
def slow_square(n):
    calls["n"] += 1
    return n * n

# your code here


### Q19 · Puzzle

Reference cycles need the garbage collector. Each `Node` adds to
`freed["n"]` when destroyed. Write `build_cyclic(n)`: a root with `n`
children, each child pointing back with `child.parent = root`; and
`build_weak(n)`, the same but `child.parent = weakref.ref(root)`.
With `gc.disable()` at the top: build and delete a cyclic tree of 1000
children and print `freed["n"]`; call `gc.collect()` and print it again.
Then reset the counter, build and delete a weak tree of 1000 children
and print `freed["n"]` without collecting.

**Expected output:**

```text
0
1001
1001
```

<details><summary><b>Solution</b></summary>

```python
import gc
import weakref

gc.disable()

def build_cyclic(n):
    root = Node()
    for _ in range(n):
        child = Node()
        child.parent = root
        root.children.append(child)
    return root

def build_weak(n):
    root = Node()
    for _ in range(n):
        child = Node()
        child.parent = weakref.ref(root)
        root.children.append(child)
    return root

tree = build_cyclic(1000)
del tree
print(freed["n"])
gc.collect()
print(freed["n"])
freed["n"] = 0
tree = build_weak(1000)
del tree
print(freed["n"])
```

</details>

In [ ]:
freed = {"n": 0}

class Node:
    def __init__(self):
        self.parent = None
        self.children = []

    def __del__(self):
        freed["n"] += 1

# your code here


---

## Q20 · Story project: Speed Up the Monday Report

Every Monday the shop's sales report takes forever. You inherit the
script: a list of `(customer_id, amount)` orders and a list of VIP
customer ids. Do not guess: **measure, change one thing, measure again.**
Each step builds on the variables and functions of the step before.

**Run these cells first.** They set up the starting files and data for every step below.

In [ ]:
import timeit

vip_ids = list(range(0, 3000, 7))
orders = [(i * 37 % 3000, 5 + (i * 13 % 90)) for i in range(20_000)]
calls = {"discount": 0}


### Step 1

Write `vip_total_slow(orders, vip_ids)` that adds up the amounts of all orders whose customer is in the **list** `vip_ids`. Print the number of orders and the VIP total.

**Expected output:**

```text
Orders: 20000
VIP total: 141262
```

<details><summary><b>Solution</b></summary>

```python
def vip_total_slow(orders, vip_ids):
    total = 0
    for customer_id, amount in orders:
        if customer_id in vip_ids:
            total += amount
    return total

print("Orders:", len(orders))
print("VIP total:", vip_total_slow(orders, vip_ids))
```

</details>

In [ ]:
# your code here


### Step 2

Write `vip_total_fast` that gives the same answer using a set. Time one call of each with `timeit.timeit(..., number=1)`. Print `True` if the totals are equal and `True` if the fast one is at least 10 times faster.

**Expected output:**

```text
True
True
```

<details><summary><b>Solution</b></summary>

```python
def vip_total_fast(orders, vip_ids):
    vip_set = set(vip_ids)
    return sum(amount for customer_id, amount in orders if customer_id in vip_set)

slow_time = timeit.timeit(lambda: vip_total_slow(orders, vip_ids), number=1)
fast_time = timeit.timeit(lambda: vip_total_fast(orders, vip_ids), number=1)
print(vip_total_slow(orders, vip_ids) == vip_total_fast(orders, vip_ids))
print(fast_time * 10 < slow_time)
```

</details>

In [ ]:
# your code here


### Step 3

Each order gets a discount from `discount_for(amount)`, which is slow and counted in `calls["discount"]`. Cache it with `functools.lru_cache`, compute the discount of every order amount, and print the real calls, then the total discount rounded to 2 decimals.

**Expected output:**

```text
Real calls: 90
Total discount: 35991.7
```

<details><summary><b>Solution</b></summary>

```python
from functools import lru_cache

def discount_for(amount):
    calls["discount"] += 1
    return amount * 0.05 if amount >= 50 else 0

discount_for = lru_cache(maxsize=None)(discount_for)
total_discount = sum(discount_for(amount) for customer_id, amount in orders)
print("Real calls:", calls["discount"])
print("Total discount:", round(total_discount, 2))
```

</details>

In [ ]:
# your code here


### Step 4

Build the report text: one line per order like `order 0: customer 0, amount 5` as a **list** of lines, then join them with newlines. Print the number of lines, the first line and the text length.

**Expected output:**

```text
Lines: 20000
order 0: customer 0, amount 5
Length: 740365
```

<details><summary><b>Solution</b></summary>

```python
lines = [f"order {i}: customer {cid}, amount {amount}" for i, (cid, amount) in enumerate(orders)]
report = "\n".join(lines)
print("Lines:", len(lines))
print(lines[0])
print("Length:", len(report))
```

</details>

In [ ]:
# your code here


### Step 5

Memory check: write `report_lines(orders)` as a generator that yields the same lines one at a time. With `tracemalloc`, measure the peak of building the full list of lines versus counting the lines from the generator. Print `True` if the counts are equal and `True` if the generator peak times 10 is smaller than the list peak.

**Expected output:**

```text
True
True
```

<details><summary><b>Solution</b></summary>

```python
import tracemalloc

def report_lines(orders):
    for i, (cid, amount) in enumerate(orders):
        yield f"order {i}: customer {cid}, amount {amount}"

def peak_of(func):
    tracemalloc.start()
    result = func()
    peak = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()
    return result, peak

list_count, list_peak = peak_of(lambda: len(list(report_lines(orders))))
stream_count, stream_peak = peak_of(lambda: sum(1 for line in report_lines(orders)))
print(list_count == stream_count)
print(stream_peak * 10 < list_peak)
```

</details>

In [ ]:
# your code here


---

## Done

Next up: **NB30 · Networking HTTP APIs** in `advanced/`.